# Rede de Citações — AI Governance in Brazilian Universities

**Projeto:** `ai-governance-brazilian-universities`  
**Insumo:** `data/processed/referencias_kwic.csv` (revisado manualmente)  

Este notebook constrói um grafo dirigido de citações a partir das referências extraídas dos 20 documentos do corpus e exporta o resultado em `.gexf` para visualização no Gephi.

**Estrutura do grafo:**
- **Nós de origem** — instituições do corpus (ex: UFPB, ANDIFES)
- **Nós de destino** — referências citadas (ex: UNESCO, LGPD, autor acadêmico)
- **Arestas** — relação de citação (origem → destino)
- **Atributos dos nós de destino** — tipo de referência (coluna `nota`), usado para colorir no Gephi
- **Tamanho dos nós de destino** — proporcional ao número de citações recebidas

---

In [ ]:
# ── Dependências ──────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import pandas as pd
import networkx as nx

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT       = Path("..")
KWIC_PATH  = ROOT / "data" / "processed" / "referencias_kwic.csv"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Ambiente pronto.")

---
## Parte 1 — Carga e filtragem dos dados

In [ ]:
df = pd.read_csv(KWIC_PATH, encoding="utf-8-sig")
df.columns = df.columns.str.strip()
# Converte colunas de texto para string, substituindo NaN por string vazia
for col in ["instituicao", "arquivo_pdf", "referencia_limpa", "valido", "nota"]:
    df[col] = df[col].fillna("").astype(str).str.strip()

print(f"Total de linhas no CSV: {len(df)}")
print(f"Colunas: {list(df.columns)}")
df.head(3)

In [ ]:
# Filtra apenas referências válidas com referencia_limpa preenchida
arestas = df[
    (df["valido"].fillna("").str.strip().str.lower() == "sim") &
    (df["referencia_limpa"].notna()) &
    (df["referencia_limpa"].fillna("").str.strip() != "")
].copy()

# Normaliza espaços
arestas["instituicao"]      = arestas["instituicao"].str.strip()
arestas["referencia_limpa"] = arestas["referencia_limpa"].str.strip()
arestas["nota"]             = arestas["nota"].fillna("").str.strip()

print(f"Referências válidas para a rede: {len(arestas)}")
print()
print("Distribuição por tipo (coluna nota):")
print(arestas["nota"].value_counts().to_string())

---
## Parte 2 — Construção do grafo

In [ ]:
# ── Grafo dirigido ────────────────────────────────────────────────────────────
G = nx.DiGraph()

# ── Nós de origem: instituições do corpus ─────────────────────────────────────
instituicoes = arestas["instituicao"].unique()
for inst in instituicoes:
    G.add_node(
        inst,
        tipo_no="corpus",
        tipo_referencia="Documento do corpus",
        tamanho=20,          # tamanho fixo para nós de origem
    )

# ── Nós de destino: referências citadas ───────────────────────────────────────
# Conta quantas vezes cada referência é citada (peso = tamanho do nó)
contagem_refs = arestas.groupby("referencia_limpa").size().reset_index(name="n_citacoes")

# Tipo de referência: pega o primeiro valor não vazio de 'nota' para cada referência
tipo_refs = (
    arestas[arestas["nota"] != ""]
    .groupby("referencia_limpa")["nota"]
    .first()
    .reset_index()
)

refs_info = contagem_refs.merge(tipo_refs, on="referencia_limpa", how="left")
refs_info["nota"] = refs_info["nota"].fillna("Não classificado")

# Normaliza tamanho: mínimo 5, máximo 50
n_min, n_max = refs_info["n_citacoes"].min(), refs_info["n_citacoes"].max()
def normaliza_tamanho(n, n_min=n_min, n_max=n_max, t_min=5, t_max=50):
    if n_max == n_min:
        return (t_min + t_max) / 2
    return t_min + (n - n_min) / (n_max - n_min) * (t_max - t_min)

for _, row in refs_info.iterrows():
    G.add_node(
        row["referencia_limpa"],
        tipo_no="referencia",
        tipo_referencia=row["nota"],
        n_citacoes=int(row["n_citacoes"]),
        tamanho=round(normaliza_tamanho(row["n_citacoes"]), 2),
    )

# ── Arestas: instituição → referência ─────────────────────────────────────────
# Agrega múltiplas citações da mesma referência pelo mesmo documento em peso
arestas_agrupadas = (
    arestas.groupby(["instituicao", "referencia_limpa", "arquivo_pdf"])
    .size()
    .reset_index(name="peso")
)

for _, row in arestas_agrupadas.iterrows():
    G.add_edge(
        row["instituicao"],
        row["referencia_limpa"],
        peso=int(row["peso"]),
        arquivo_pdf=row["arquivo_pdf"],
    )

print(f"Nós:   {G.number_of_nodes()}")
print(f"Arestas: {G.number_of_edges()}")
print()
print(f"Nós de origem (corpus):     {len(instituicoes)}")
print(f"Nós de destino (referências): {len(refs_info)}")

---
## Parte 3 — Análise descritiva da rede

In [ ]:
# Referências mais citadas
mais_citadas = (
    refs_info
    .sort_values("n_citacoes", ascending=False)
    .head(20)
    .reset_index(drop=True)
)
mais_citadas.index += 1
print("Top 20 referências mais citadas:")
print(mais_citadas[["referencia_limpa", "nota", "n_citacoes"]].to_string())

In [ ]:
# Instituições com mais referências únicas
refs_por_inst = (
    arestas.groupby("instituicao")["referencia_limpa"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index()
)
refs_por_inst.columns = ["instituicao", "refs_unicas"]
refs_por_inst.index += 1
print("Referências únicas por instituição:")
print(refs_por_inst.to_string())

In [ ]:
# Referências compartilhadas por mais de uma instituição
refs_compartilhadas = (
    arestas.groupby("referencia_limpa")["instituicao"]
    .nunique()
    .reset_index()
)
refs_compartilhadas.columns = ["referencia_limpa", "n_instituicoes"]
refs_compartilhadas = (
    refs_compartilhadas[refs_compartilhadas["n_instituicoes"] > 1]
    .sort_values("n_instituicoes", ascending=False)
    .reset_index(drop=True)
)
refs_compartilhadas.index += 1
print(f"Referências citadas por mais de uma instituição: {len(refs_compartilhadas)}")
print()
print(refs_compartilhadas.to_string())

---
## Parte 4 — Exportação para Gephi (.gexf)

O formato `.gexf` preserva todos os atributos dos nós e arestas.
No Gephi, use a coluna `tipo_referencia` para colorir os nós e `tamanho` para dimensioná-los.

In [ ]:
# Sanitiza atributos dos nós — remove caracteres problemáticos
for node, attrs in G.nodes(data=True):
    for key, val in attrs.items():
        if isinstance(val, str):
            G.nodes[node][key] = val.replace("&", "e").replace("<", "").replace(">", "")

# Sanitiza atributos das arestas
for u, v, attrs in G.edges(data=True):
    for key, val in attrs.items():
        if isinstance(val, str):
            G[u][v][key] = val.replace("&", "e").replace("<", "").replace(">", "")

In [ ]:
saida_gexf = OUTPUT_DIR / "rede_citacoes.gexf"
nx.write_gexf(G, saida_gexf)
print(f"Grafo exportado: {saida_gexf}")
print(f"Nós: {G.number_of_nodes()} | Arestas: {G.number_of_edges()}")
print()
print("Como abrir no Gephi:")
print("  1. File → Open → seleciona rede_citacoes.gexf")
print("  2. Appearance → Nodes → Color → Partition → tipo_referencia")
print("  3. Appearance → Nodes → Size → Ranking → tamanho")
print("  4. Layout → ForceAtlas 2 → Run")

In [ ]:
saida_graphml = OUTPUT_DIR / "rede_citacoes.graphml"
nx.write_graphml(G, saida_graphml)
print(f"Grafo exportado: {saida_graphml}")
print(f"Nós: {G.number_of_nodes()} | Arestas: {G.number_of_edges()}")
print()
print("Como abrir no Gephi:")
print("  1. File → Open → seleciona rede_citacoes.graphml")
print("  2. Appearance → Nodes → Color → Partition → tipo_referencia")
print("  3. Appearance → Nodes → Size → Ranking → tamanho")
print("  4. Layout → ForceAtlas 2 → Run")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Cores por tipo de referência ──────────────────────────────────────────────
CORES = {
    "Documento do corpus":    "#2D5FA8",
    "Oficial nacional":       "#1A8C6A",
    "Oficial internacional":  "#C0392B",
    "Acadêmico nacional":     "#E67E22",
    "Acadêmico internacional":"#8E44AD",
    "Não classificado":       "#AAAAAA",
}

# ── Layout ────────────────────────────────────────────────────────────────────
pos = nx.spring_layout(G, seed=42, k=2.5)

# ── Atributos visuais dos nós ─────────────────────────────────────────────────
node_colors = [CORES.get(G.nodes[n].get("tipo_referencia", ""), "#AAAAAA") for n in G.nodes]
node_sizes  = [G.nodes[n].get("tamanho", 10) * 20 for n in G.nodes]

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 12))

nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.3, edge_color="#CCCCCC",
                       arrows=True, arrowsize=8, width=0.7)

nx.draw_networkx_nodes(G, pos, ax=ax,
                       node_color=node_colors,
                       node_size=node_sizes,
                       alpha=0.9)

nx.draw_networkx_labels(G, pos, ax=ax, font_size=7, font_color="#222222")

# ── Legenda ───────────────────────────────────────────────────────────────────
handles = [mpatches.Patch(color=cor, label=tipo) for tipo, cor in CORES.items()]
ax.legend(handles=handles, loc="upper left", fontsize=9, framealpha=0.9)

ax.set_title("Rede de citações — AI Governance in Brazilian Universities",
             fontsize=14, fontweight="bold", pad=16)
ax.axis("off")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "rede_citacoes.png", dpi=150, bbox_inches="tight")
plt.show()